# 6. Explore annotated mouse-bladder MSI data in PRIDE

This notebook searches metadata only. It displays accepted image pairs, remote sizes, annotation compatibility, rejected records, and PRIDE links before MSI data are downloaded. The exported JSON is consumed by the existing `query --filters` command.

In [ ]:
from pathlib import Path

from msi_autoencoder_wrapper.dataset_management.exploration import DatasetExplorer

explorer = DatasetExplorer(source="pride")

## Inspect and configure filters

The source reports supported fields and their meaning. The configuration requires unambiguous organism and organ metadata, a complete imzML/ibd pair, a supported curated molecule-to-pixel table, and downloadable checksums.

In [ ]:
explorer.available_filters()

In [ ]:
filters = {
    "keyword": "imzML bladder",
    "organisms": ["Mus musculus (mouse)"],
    "organism_parts": ["Urinary bladder", "Bladder"],
    "experiment_types": ["mass spectrometry imaging"],
    "required_metadata_fields": ["organisms", "organismParts"],
    "single_value_metadata_fields": ["organisms", "organismParts", "diseases"],
    "require_annotation_source": True,
    "require_checksum": True,
    "page_size": 100,
    "max_projects": 500,
    "exclude_dataset_ids": [],
}
explorer.set_filters(filters)

## Search and review

Accepted rows are independent MSI image pairs. `total_size_bytes` is the imzML plus ibd size reported before download. Open `project_url` to inspect the source. Rejected rows explain missing metadata, incomplete pairs, unavailable checksums, or unsupported annotations.

In [ ]:
accepted = explorer.search()
accepted

In [ ]:
rejected = explorer.rejected()
rejected

## Exclude reviewed images

After opening project links, place unwanted `dataset_id` values below. Exclusions become part of the exported query configuration.

In [ ]:
excluded_dataset_ids = []
if excluded_dataset_ids:
    explorer.exclude(excluded_dataset_ids)
explorer.results()

In [ ]:
output_path = Path("assets/configs/datasets/pride_mouse_bladder.json")
explorer.export_config(output_path)

## Use the exported configuration

Run `manage_datasets.py query --source pride --filters assets/configs/datasets/pride_mouse_bladder.json --selection workspace/datasets/selections/pride_mouse_bladder.json`, review the selection, and then pass that selection to `download --source pride`.